# Content/SEO Agent — Model Fine-Tune

Fine-tunes a small instruction-tuned model on the classification + drafting tasks using LoRA (Unsloth), then exports to GGUF for CPU inference via Ollama.

**Before running:** Runtime -> Change runtime type -> T4 GPU (free tier is fine for a ~1.5B model).

**Before this notebook:** run `training/prepare_training_data.py` against your own product export to generate `combined_train.jsonl` and `combined_val.jsonl`. Every prompt in those files is built from your `config.yaml` automatically -- this notebook needs no company-specific edits.

## 1. Install dependencies

In [1]:
%%capture
!pip install unsloth

**Use a genuinely fresh runtime for this.** Runtime -> Disconnect and delete runtime, reconnect with T4 GPU, then run this cell before anything else. If you already ran a different install in this session, restarting alone is not enough -- delete and reconnect the runtime.

## 2. Upload training files

In [ ]:
from google.colab import files
print("Upload combined_train.jsonl and combined_val.jsonl (from training/prepare_training_data.py)")
uploaded = files.upload()

## 3. Load base model (4-bit for fast Colab training)

In [ ]:
from unsloth import FastLanguageModel
import torch

# Swap this for a different small instruction-tuned model if you prefer --
# anything in the 1-4B range works well on a free-tier T4. Update the
# chat_template in the cell below to match if you change the model family.
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"
CHAT_TEMPLATE = "qwen-2.5"

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = max_seq_length,
    dtype = None,          # auto-detect (bf16 on T4-supported cards, fp16 otherwise)
    load_in_4bit = True,
)

## 4. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 5. Load and format the dataset (ChatML template)

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template = CHAT_TEMPLATE)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
             for convo in convos]
    return {"text": texts}

dataset = load_dataset("json", data_files={
    "train": "combined_train.jsonl",
    "validation": "combined_val.jsonl",
})
dataset = dataset.map(formatting_prompts_func, batched=True)

print(dataset["train"][0]["text"][:800])

## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["validation"],
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 20,
        eval_strategy = "steps",
        eval_steps = 100,
        save_strategy = "no",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

## 7. Quick sanity check
Test both task types on a held-out example before exporting. Paste in your own `config.yaml`'s system prompts here for an accurate check -- the placeholder below is illustrative only and won't exactly match what your model was actually trained on unless your config uses these exact values.

In [ ]:
FastLanguageModel.for_inference(model)

# Replace this with your actual settings.SYSTEM_CLASSIFY value (printed by
# `python -c "from config import settings; print(settings.SYSTEM_CLASSIFY)"`)
# for an exact match to what the model was trained on.
example_system_prompt = (
    "You are a content classification assistant for [Your Company], [your business "
    "description]. Given a product title and its current description (which may be "
    "empty), output ONLY a JSON object with these fields: "
    '{"is_regulated": true/false, "content_status": "missing"|"thin"'
    '|"plain_text_needs_formatting"|"regulated_missing_disclaimer"|"good", '
    '"product_type": "bulk_bag"|"roll"|"bagged"|"each_or_pack"|"other"}.'
)

test_messages = [
    {"role": "system", "content": example_system_prompt},
    {"role": "user", "content": "Product title: [paste a real title from your own catalog here]\nCurrent description: (no description)"},
]

inputs = tokenizer.apply_chat_template(test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=150, use_cache=True)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

## 8. Merge LoRA into base weights and export to GGUF (CPU inference format)
`q4_k_m` is a good balance of size vs quality for CPU inference.

In [ ]:
model.save_pretrained_gguf(
    "content-agent-gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)

## 9. Save to Google Drive and download
`files.download()` is unreliable for files this large (~1GB) as it streams through browser memory. Routing through Drive is more robust: this cell mounts your Drive and copies the file there, then you download it normally from drive.google.com.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, glob
gguf_files = glob.glob("**/*.gguf", recursive=True)
print("Found:", gguf_files)

dest = "/content/drive/MyDrive/content_agent.gguf"
shutil.copy(gguf_files[0], dest)
print(f"Copied to {dest}")

## 10. Deploy with Ollama

After downloading `content_agent.gguf`, on your serving machine:

```
FROM /path/to/content_agent.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
"""

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.3
```

Save as `Modelfile`, then:
```
ollama create <name-matching-config.yaml-models.small_model_name> -f Modelfile
```